In [8]:
from pathlib import Path
import numpy as np
import torch
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim

from mads_datasets import datatools
import mltrainer
mltrainer.__version__

'0.2.4'

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. test


In [9]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2025-09-25 20:12:26.796 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\r.weenink\.cache\mads_datasets\gestures
100%|██████████| 651/651 [00:00<00:00, 1805.09it/s]


In [107]:
#get the maximum length of the sequences in the training set
max_len = max([len(x[0]) for x in train.dataset])
max_len

51

In [111]:
#get the maximum length of the x[1]
max_len = max([len(x[1]) for x in train.dataset])

TypeError: len() of a 0-d tensor

In [10]:
len(train), len(valid)

(81, 20)

In [66]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 32, 3]),
 tensor([ 8,  7, 16, 14, 14,  1, 19,  7,  4, 18,  2,  2, 19, 16, 16,  4,  2,  8,
          1, 12,  3,  4, 15, 18, 18, 10,  6, 17, 15, 10, 17, 11]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?

batch, amount of gestures, accelaration in 3 directions


# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

The 2nd number changes.  This mus be resolved By giving it padding. So the data is not padded.

In [73]:
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [87]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=150,
    num_layers=3,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [88]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [89]:
accuracy(y, yhat)

0.0625

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [45]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [41]:
yhat[22]

tensor([-0.0247,  0.0977, -0.1977, -0.2393,  0.1562,  0.0352, -0.0986,  0.0691,
         0.0042,  0.2813, -0.1834, -0.2023,  0.0310, -0.1746, -0.0951, -0.1091,
         0.0881, -0.1020,  0.2398, -0.0602], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

I think the lower the number the least the chance it is the class. Because tanh this is the case.
In this case we get the class of item 0. 

In [42]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(2.9786, grad_fn=<NllLossBackward0>)

In [43]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

using cpu


Set up the settings for the trainer and the different types of logging you want

In [97]:
settings = TrainerSettings(
    epochs=25, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings

epochs: 25
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 1e-05, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

In [ ]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        
        self.linear = nn.Sequential(
            nn.pad
            nn.Linear(config.hidden_size, config.output_size))

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [99]:
config = ModelConfig(
    input_size=3,
    hidden_size=150,
    num_layers=1,
    output_size=20,
    dropout=0.1,
)


In [100]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "modelname-here")
    mlflow.set_tag("dev", "your-name-here")
    config = ModelConfig(
        input_size=3,
        hidden_size=150,
        num_layers=2,
        output_size=20,
        dropout=0.2,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2025-09-25 20:46:44.018 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20250925-204644
2025-09-25 20:46:44.019 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:05<00:00, 13.79it/s]
2025-09-25 20:46:50.310 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.9920 test 2.9921 metric ['0.0750']
100%|██████████| 81/81 [00:05<00:00, 15.35it/s]
2025-09-25 20:46:55.954 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.9875 test 2.9866 metric ['0.0719']
100%|██████████| 81/81 [00:05<00:00, 14.77it/s]
2025-09-25 20:47:01.916 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 2.9828 test 2.9823 metric ['0.0969']
100%|██████████| 81/81 [00:05<00:00, 14.50it/s]
2025-09-25 20:47:08.328 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 2.9769 test 2.9777 metric ['0.0875']
100%|██████████| 81/81 [00:20<00:00,  3.92it/s]
2025-09-25 20:47:30.82

Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

In [24]:
trainer.loop() # if you want to pick up training, loop will continue from the last epoch

100%|██████████| 81/81 [00:01<00:00, 43.75it/s]
2025-09-25 20:12:46.720 | INFO     | mltrainer.trainer:report:175 - Resuming epochs from previous training at 3
2025-09-25 20:12:46.810 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 2.9936 test 2.9923 metric ['0.0484']
100%|██████████| 81/81 [00:02<00:00, 31.84it/s]
2025-09-25 20:12:49.550 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 2.9934 test 2.9931 metric ['0.0500']
2025-09-25 20:12:49.553 | INFO     | mltrainer.trainer:__call__:252 - best loss: 2.9923, current loss 2.9931.Counter 1/5.
100%|██████████| 81/81 [00:02<00:00, 32.04it/s]
2025-09-25 20:12:52.280 | INFO     | mltrainer.trainer:report:209 - Epoch 5 train 2.9915 test 2.9880 metric ['0.0578']
100%|██████████| 3/3 [00:07<00:00,  2.51s/it]


In [25]:
mlflow.end_run()

Excercises:

- try to improve the RNN model
- test different things. What works? What does not?
- experiment with either GRU or LSTM layers, create your own models. Have a look at `mltrainer.rnn_models` for inspiration. 
- experiment with adding Conv1D layers. Think about the necessary input-output dimensions of your tensors before and after each layer.

You should be able to get above 90% accuracy with the dataset.
Create a report of 1 a4 about your experiments.